## Step 1: Import Libraries and Keys

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import json

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None: 
    raise Exception("OpenAI API Key is not set!")

client = OpenAI(api_key=OPENAI_API_KEY)



## Step 2: Setup Pushover
 
 - Setup account in the browser: done!
 - Setup app on phone, login to same account: done!
 - In browser, create Application/API Token: done!
 - Copy tokens into the .env file

 

In [2]:
load_dotenv()

True

In [3]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [4]:
#Test Pusheover
import requests

def send_notification(message: str):
    payload = {'user': pushover_user, 'token': pushover_token, 'device': 'oneplusnordn2005g', 'message': message}
    requests.post(pushover_url, data=payload)


In [ ]:
send_notification("Hola Barbara, you are the best!")

## Step 3: Describe Pushover as an LLM tool


In [5]:
send_notification_function = {
    'name': 'send_notification',
    'description': """
                Sends a message to send as a push notification to the users phone via Pushover. 
                User this to alert the user about a new message.""",
    'parameters': {
        'type': 'object',
        'properties': {
            'message': {
                'type': 'string',
                'description': 'The notification message to send to the users device'
            }
        },
        "required": ["message"]
    }
}

## Step 4: Add Pushover to the list of tools for the LLM

In [6]:
tools = [{"type":"function", "function": send_notification_function}]

## Step 5: Calling the tools from an LLM

In [18]:
def handle_tool_call(tool_calls):
    tool_call = tool_calls[0] #assuming only 1 tool call
    args = json.loads(tool_call.function.arguments)
    send_notification(args['message'])
    
    tool_call_result = {
        "role": "tool",
        "content": f"We sent a push notification: {args['message']}",
        "tool_call_id": tool_call.id

    }
    return tool_call_result

In [ ]:
client = OpenAI()
messages = [
    {"role": "user", "content": """
                Please send me a notification that tells me to believe in myself because I am amazing and tell me why in a few words"""}
                
]

response = client.chat.completions.create(
    model = 'gpt-4.1-mini',
    messages = messages,
    temperature=0.9,
    tools=tools
)

#Check if model wants to call a tool
resp = response.choices[0].message
# print(message)

if resp.tool_calls:
    # handle tool call
    tool_result = handle_tool_call(resp.tool_calls)

    messages.append(resp)
    messages.append(tool_result)

    # invoke the LLM one more time to get updated response
    response = client.chat.completions.create(
        model = 'gpt-4.1-mini',
        messages = messages
    )

# print(response of LLM to user)
print(response.choices[0].message.content)

I've sent you a notification that says: "Believe in yourself because you are amazing! Your unique talents and resilience make you capable of achieving great things." Keep shining!


## Week 3, Day 5 - Multiple Tools Called


In [ ]:
def handle_tool_call(tool_calls):
    tool_results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name 
        args = json.loads(tool_call.function.arguments)
        print(f"Calling fucnction {function_name}") #For future debuggins

        #Route to the appropriate function based on function_name
        if function_name == "send_notification":
            send_notification(args['message'])
            content = f"Sent na notification: {args['message']}"
        elif function_name == 'inspire_emoji':
            content = convert_message_to_emoji(args['message'])
        elif function_name == 'dice_roll':
            content = f"Dice roll was: {dice_roll()}"
        else:
            content = f"Unknown function: {function_name}"
        
        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }
        tool_results.append(tool_call_result)
    return tool_results

client = OpenAI()
messages = [
    {"role": "user", "content": """
                Please send me 2 notifications that tells me to believe in myself because (1) Tell me that I am amazing, then WAIT 1 SECOND and next (2) Send another notification telling me a rhyme"""}
                
]

response = client.chat.completions.create(
    model = 'gpt-4.1-mini',
    messages = messages,
    temperature=0.9,
    tools=tools
)

if response.choices[0].message.tool_calls:
    # handle tool call
    tool_result = handle_tool_call(response.choices[0].message.tool_calls)

    messages.append(response.choices[0].message)
    messages.extend(tool_result)

    # invoke the LLM one more time to get updated response
    response = client.chat.completions.create(
        model = 'gpt-4.1-mini',
        messages = messages
    )

# print(response of LLM to user)
print(response.choices[0].message.content)

Notification 1: Believe in yourself because you are amazing!

(Waited 1 second)

Notification 2: Believe in yourself, stand tall and fine, the world is bright because you shine!
